# Notebook 12 — Power Analysis and Sample Size

**Module:** 03 — Statistics and Probability  
**Notebook:** 12 of 18  
**Prerequisites:** NB05, NB06  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

An underpowered study is a waste of resources: it cannot detect a true effect even if
one exists. Funding bodies and ethics committees increasingly require power calculations
before experiments begin. For RNA-seq: how many replicates do I need to detect a 2-fold
change with 80% power? This notebook answers that question from first principles.

---
## Step 3 — Biological Background

**RNA-seq design:**
- Biological replicates (different samples) are essential for power calculations
- Technical replicates (sequencing same sample twice) do not add statistical power
- Typical RNA-seq: n=3 per group is underpowered for small effects; n=6–10 recommended
- Power depends on: effect size (fold change), noise level (CV), number of replicates

**The four-way relationship:**
Fix any three → the fourth is determined:
1. $\alpha$ (significance threshold)
2. $\beta$ (Type II error rate; power = 1-β)
3. Effect size $d$ (Cohen's d or fold change)
4. $n$ (sample size per group)

---
## Step 4 — Mathematical Explanation

**Power for a two-sample t-test:**

Under $H_1$ with effect size $d = (\mu_1 - \mu_2)/\sigma$, the non-centrality
parameter is:
$$\delta = d \cdot \sqrt{n/2}$$

The power is:
$$\text{Power} = P(|T| > t_{\alpha/2, 2n-2} \mid \delta)$$
where $T$ follows a non-central t-distribution with df=2n-2 and non-centrality $\delta$.

**Normal approximation for sample size:**
For large n, the required sample size per group is approximately:
$$n \approx \frac{2(z_{\alpha/2} + z_\beta)^2}{d^2}$$
where $z_{\alpha/2}$ and $z_\beta$ are standard Normal quantiles.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Power calculation from scratch (non-central t distribution)
def compute_power(n: int, effect_size_d: float, alpha: float = 0.05) -> float:
    """
    Power of a two-sample Welch t-test.

    Parameters
    ----------
    n : sample size per group
    effect_size_d : Cohen's d
    alpha : significance threshold (two-tailed)

    Returns
    -------
    power : float in [0, 1]
    """
    df = 2 * n - 2
    ncp = effect_size_d * np.sqrt(n / 2)  # non-centrality parameter
    crit = stats.t.ppf(1 - alpha/2, df=df)  # critical value
    # Power = P(|T| > crit | H1) using non-central t
    nc_t = stats.nct(df=df, nc=ncp)
    power = nc_t.sf(crit) + nc_t.cdf(-crit)
    return power

def sample_size_for_power(target_power: float, effect_size_d: float,
                            alpha: float = 0.05) -> int:
    """
    Minimum n per group for target power (Normal approximation).
    """
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(target_power)
    n_approx = 2 * (z_alpha + z_beta)**2 / effect_size_d**2
    return int(np.ceil(n_approx))

# Power table
print(f"{'n per group':>12} {'d=0.2':>8} {'d=0.5':>8} {'d=0.8':>8} {'d=1.2':>8}")
print("-" * 48)
for n in [5, 10, 20, 30, 50, 100]:
    row = [compute_power(n, d) for d in [0.2, 0.5, 0.8, 1.2]]
    print(f"  {n:>10}  " + "  ".join(f"{p:>6.2f}" for p in row))

print(f"\nRequired n per group for 80% power:")
for d in [0.2, 0.5, 0.8]:
    n = sample_size_for_power(0.80, d)
    print(f"  d={d}: n={n}")

In [ ]:
# Cell 6.2 — RNA-seq specific: power for log2 fold change
def rnaseq_cohen_d(log2fc: float, cv: float = 0.4) -> float:
    """
    Cohen's d from log2 fold change and coefficient of variation.

    Assumes log-normal data: sigma_log ≈ cv (valid for small CV).
    """
    return log2fc / cv

print("RNA-seq power analysis (CV=0.4, α=0.05):")
print(f"{'log2FC':>8} {'Cohen\'s d':>10} {'n for 80% power':>18}")
print("-" * 40)
for log2fc in [0.5, 1.0, 1.5, 2.0]:
    d = rnaseq_cohen_d(log2fc, cv=0.4)
    n = sample_size_for_power(0.80, d)
    print(f"  {log2fc:>6.1f}    {d:>9.2f}   {n:>15}")

print("\nInterpretation: detecting 2-fold changes (log2FC=1) requires ~6 reps at CV=0.4")

In [ ]:
# Cell 6.3 — Validate power calculation by simulation
rng = np.random.default_rng(42)
d_test, n_test = 0.5, 20
n_sim = 5000

theoretical_power = compute_power(n_test, d_test)
detected = sum(
    stats.ttest_ind(rng.normal(0,1,n_test), rng.normal(d_test,1,n_test))[1] < 0.05
    for _ in range(n_sim)
)
simulated_power = detected / n_sim

print(f"d={d_test}, n={n_test} per group:")
print(f"  Theoretical power: {theoretical_power:.4f}")
print(f"  Simulated power:   {simulated_power:.4f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Power curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: power vs. n for different effect sizes
ax = axes[0]
n_range = np.arange(3, 101)
for d, color in [(0.2, "steelblue"), (0.5, "green"), (0.8, "orange"), (1.2, "tomato")]:
    powers = [compute_power(n, d) for n in n_range]
    ax.plot(n_range, powers, lw=2, color=color, label=f"d={d}")
ax.axhline(0.8, color="black", ls="--", lw=1, label="80% power")
ax.set_xlabel("n per group"); ax.set_ylabel("Power (1-β)")
ax.set_title("Power curves: two-sample t-test (α=0.05)")
ax.legend(fontsize=8)

# Panel 2: required n vs. effect size for 80% power
ax = axes[1]
d_range = np.linspace(0.1, 2.0, 100)
n_required = [sample_size_for_power(0.80, d) for d in d_range]
ax.plot(d_range, n_required, lw=2, color="steelblue")
# Annotate RNA-seq log2FC values
for log2fc, label in [(1.0, "2-fold"), (1.5, "2.8-fold"), (2.0, "4-fold")]:
    d_annot = rnaseq_cohen_d(log2fc, cv=0.4)
    n_annot = sample_size_for_power(0.80, d_annot)
    ax.annotate(f"{label}\nn={n_annot}", (d_annot, n_annot),
                xytext=(d_annot + 0.15, n_annot + 5), fontsize=8,
                arrowprops=dict(arrowstyle="->", color="gray"))
ax.set_xlabel("Cohen's d"); ax.set_ylabel("Required n per group")
ax.set_title("Sample size for 80% power (α=0.05)")
ax.set_ylim(0, 200)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `compute_power()` from scratch using the non-central t distribution.
   Verify against the simulation in Cell 6.3 for d=0.8, n=15.
2. An RNA-seq experiment detects a 1.5-fold change (log2FC=0.585) with CV=0.6.
   How many replicates per group are needed for 80% power at FDR=0.05
   (accounting for multiple testing with ~10,000 genes)?
3. What is the relationship between effect size and required n? Derive the approximate
   $n \propto 1/d^2$ relationship from the sample size formula.
4. Why does increasing statistical power also increase the cost of a study? What is
   the ethical justification for requiring power calculations before animal experiments?

---
## Quiz — Active Recall

1. What are the four quantities that determine statistical power? Write the formula.
2. What is the non-centrality parameter? What does it represent geometrically?
3. Why is n=3 per group often insufficient for RNA-seq experiments?
4. If you double the effect size, by approximately what factor does required n change?
5. What happens to power when you apply Bonferroni correction to 20,000 tests?

---
## Reflection

**Date completed:** ____________________

> *[Could you advise a collaborator on how many RNA-seq replicates they need? What additional information would you need from them?]*

---
*Next: `13_bayesian_statistics_intuition.ipynb`*